# Multi-Modal RAG with Smart Parsing + Advanced Indexing

This notebook demonstrates **multi-modal RAG** using a Bedrock Managed Knowledge Base. It combines **Smart Parsing** (the text extraction engine) with **Advanced Indexing** (media extraction) to ingest and query across PDFs with charts, audio recordings, and video content.

### Smart Parsing + Advanced Indexing

| Component | What it does |
|-----------|------|
| **Smart Parsing** | Always on — extracts text from all supported file types (PDF, DOCX, HTML, etc.) |
| **Advanced Indexing** | Opt-in — additionally extracts images from documents, transcribes audio, processes video |
| **Managed Embedding** | Embeds all extracted content into the vector store (no extra cost) |

Without Advanced Indexing, only text is extracted. With it enabled, charts/diagrams in PDFs, audio transcriptions, and video content are all indexed and searchable.

### Multi-modal content in this notebook

| File | Type | What gets extracted |
|------|------|----------------------------|
| `IF12695.pdf` | PDF with charts & tables | Text + chart/figure descriptions (via image extraction) |
| `podcastdemo.mp3` | Audio | Transcribed speech → searchable text (via audio extraction) |
| `bda.m4v` | Video | Video content descriptions (via video extraction) |

### Architecture

```
┌─────────────────┐   ┌───────────┐   ┌───────────┐
│ PDF (charts)    │   │ Audio     │   │ Video     │
│ IF12695.pdf     │   │ .mp3      │   │ .m4v      │
└──────┬──────────┘   └─────┬─────┘   └─────┬─────┘
       │                    │               │
       └────────────────────┼───────────────┘
                            ▼
              ┌──────────────────────────────┐
              │     Smart Parsing (always on) │
              │     + Advanced Indexing        │
              │                                │
              │  imageExtractionStatus: ENABLED │ ← charts/diagrams from PDF
              │  audioExtractionStatus: ENABLED │ ← transcribes MP3
              │  videoExtractionStatus: ENABLED │ ← processes M4V
              └──────────────┬─────────────────┘
                             ▼
              ┌──────────────────────────────┐
              │   Managed Embedding (default) │ ← No extra cost
              └──────────────┬───────────────┘
                             ▼
              ┌──────────────────────────────┐
              │   Managed Vector Store        │
              └──────────────┬───────────────┘
                             ▼
               Retrieve / AgenticRetrieveStream
```

### Why use Advanced Indexing?

| Without Advanced Indexing | With Advanced Indexing |
|---|---|
| Only text from PDFs indexed | Text + chart/diagram descriptions indexed |
| Audio files may not be transcribed | Audio fully transcribed and searchable |
| Video files not processed | Video content extracted and indexed |
| "What does the chart show?" → no answer | "What does the chart show?" → detailed answer |

### Prerequisites

- AWS credentials with Bedrock, IAM, and S3 permissions
- Model access enabled for **Managed Embedding** and **Claude**
- Python 3.10+

In [ ]:
%pip install --upgrade pip --quiet
%pip install -r ../../requirements.txt --quiet

In [ ]:
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Step 1 — Configuration

In [ ]:
import boto3
import sys
import time
import os
import json
import logging
import pprint

sys.path.insert(0, "../..")

s3_client = boto3.client('s3')
sts_client = boto3.client('sts')
session = boto3.session.Session()
region = session.region_name
account_id = sts_client.get_caller_identity()['Account']

logging.basicConfig(format='[%(asctime)s] p%(process)s {%(filename)s:%(lineno)d} %(levelname)s - %(message)s', level=logging.INFO)
logger = logging.getLogger(__name__)

suffix = time.strftime('%Y%m%d%H%M%S', time.localtime())[-7:]

# ── Configuration ─────────────────────────────────────────────────────
knowledge_base_name = f'bmkb-multimodal-{suffix}'
bucket_name = f'bedrock-bmkb-multimodal-{suffix}-{account_id}'

# Embedding model — use managed default (no extra cost)
# Advanced Indexing converts all media to text, so text embeddings work perfectly
embedding_model = None

# To use a custom embedding model (additional cost), uncomment:
# embedding_model = 'amazon.titan-embed-text-v2:0'

# Generation model for AgenticRetrieveStream
region_prefix_map = {'us-': 'us', 'eu-': 'eu', 'ap-': 'apac'}
cris_prefix = next((v for k, v in region_prefix_map.items() if region.startswith(k)), 'us')
generation_model_arn = f'arn:aws:bedrock:{region}:{account_id}:inference-profile/{cris_prefix}.anthropic.claude-haiku-4-5-20251001-v1:0'

pp = pprint.PrettyPrinter(indent=2)

print(f'Region:     {region}')
print(f'Account:    {account_id}')
print(f'KB Name:    {knowledge_base_name}')
print(f'Bucket:     {bucket_name}')
print(f'Embedding:  {embedding_model or "managed default (no extra cost)"}')
print(f'Generation: {generation_model_arn}')

## Step 2 — Prepare multi-modal data

Upload three types of content:
1. **PDF with charts/tables** — CRS report with visual elements (charts, figures, data tables)
2. **Audio** — Podcast demo (MP3)
3. **Video** — BDA demo video (M4V)

In [ ]:
# Create S3 bucket
try:
    s3_client.head_bucket(Bucket=bucket_name)
    print(f'Bucket {bucket_name} already exists')
except Exception:
    print(f'Creating bucket {bucket_name}')
    if region == 'us-east-1':
        s3_client.create_bucket(Bucket=bucket_name)
    else:
        s3_client.create_bucket(Bucket=bucket_name, CreateBucketConfiguration={'LocationConstraint': region})

In [ ]:
# CRS report PDF — contains charts, tables, and visual content
# This file must be downloaded manually (the site blocks programmatic access).
#
# Download from your browser: https://sgp.fas.org/crs/misc/IF12695.pdf
# Save to: ../../synthetic_dataset/IF12695.pdf
#
# If you want to use a different PDF with visual content, update the path below.

crs_pdf_path = '../../synthetic_dataset/IF12695.pdf'

if os.path.exists(crs_pdf_path) and os.path.getsize(crs_pdf_path) > 1000:
    size_kb = os.path.getsize(crs_pdf_path) / 1024
    print(f'✓ CRS report ready: {crs_pdf_path} ({size_kb:.1f} KB)')
else:
    print('⚠️  CRS report not found or empty.')
    print('    Download from: https://sgp.fas.org/crs/misc/IF12695.pdf')
    print(f'    Save to: {os.path.abspath(crs_pdf_path)}')
    print('    Then re-run this cell.')

In [ ]:
# Upload all multi-modal content to S3
multimodal_files = [
    '../../synthetic_dataset/IF12695.pdf',           # PDF with charts and tables
    '../../synthetic_dataset/podcastdemo.mp3',       # Audio file
    '../../synthetic_dataset/bda.m4v',               # Video file
]

print('Uploading multi-modal content to S3...')
for file_path in multimodal_files:
    if os.path.exists(file_path):
        file_name = os.path.basename(file_path)
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f'  {file_name} ({size_mb:.1f} MB) → s3://{bucket_name}/{file_name}')
        s3_client.upload_file(file_path, bucket_name, file_name)
    else:
        print(f'  ⚠️  {file_path} not found — skipping')

# List uploaded files
print(f'\nFiles in bucket:')
resp = s3_client.list_objects_v2(Bucket=bucket_name)
for obj in resp.get('Contents', []):
    ext = obj['Key'].split('.')[-1].upper()
    size_mb = obj['Size'] / (1024 * 1024)
    modality = {'PDF': '📄', 'MP3': '🎵', 'M4V': '🎬', 'MP4': '🎬'}.get(ext, '📎')
    print(f'  {modality} {obj["Key"]} ({size_mb:.1f} MB)')

## Step 3 — Create BMKB with Advanced Indexing

We enable all three media extraction toggles:
- `enable_image_extraction=True` — extracts and describes charts/diagrams from the PDF
- `enable_audio_extraction=True` — transcribes the MP3 podcast
- `enable_video_extraction=True` — processes the M4V video

This is configured via `mediaExtractionConfiguration` in the data source, alongside the always-on Smart Parsing.

In [ ]:
from utils.managed_knowledge_base import ManagedKnowledgeBase

kb = ManagedKnowledgeBase(
    kb_name=knowledge_base_name,
    data_sources=[{
        'type': 'S3',
        'bucket_name': bucket_name,
        # Advanced Indexing — enable all media extraction
        'enable_image_extraction': True,   # Charts/diagrams from PDF
        'enable_audio_extraction': True,   # Transcribe MP3
        'enable_video_extraction': True,   # Process M4V video
    }],
    embedding_model=embedding_model,
    enable_logging=True,
    region_name=region,
    suffix=suffix,
)

print(f'\nKB ID: {kb.kb_id}')
print(f'DS ID: {kb.ds_id}')
print(f'\nAdvanced Indexing enabled:')
print(f'  Image extraction: ✓ (charts/diagrams from PDF)')
print(f'  Audio extraction: ✓ (transcribes MP3)')
print(f'  Video extraction: ✓ (processes M4V)')

## Step 4 — Ingest multi-modal content

Ingestion with Advanced Indexing takes longer than text-only because it processes:
- PDF → text extraction + image/chart description generation
- MP3 → audio transcription
- M4V → video analysis

In [ ]:
time.sleep(30)
job = kb.start_ingestion_job()

## Step 5 — Query visual content from PDF

These queries target content that would only be available with image extraction enabled.
Without Advanced Indexing, charts/figures in the PDF are invisible to retrieval.

In [ ]:
# Queries that specifically target visual/chart content in the PDF
visual_queries = [
    'Describe the data shown in the charts and figures in the report.',
    'What trends or patterns are illustrated in the graphs?',
    'What numerical data is presented in the tables and visual elements?',
]

print('=== Visual Content Queries (PDF charts/figures) ===')
print('These results require image extraction to capture chart descriptions.\n')

for query in visual_queries:
    print(f'Q: {query}')
    response = kb.retrieve(query, num_results=3)
    for i, res in enumerate(response.get('retrievalResults', []), 1):
        score = res.get('score', 0)
        text = res['content']['text'][:180]
        uri = res.get('location', {}).get('s3Location', {}).get('uri', '')
        source = uri.split('/')[-1] if uri else ''
        print(f'  {i}. score={score:.4f} | source={source}')
        print(f'     {text}...')
    print()

## Step 6 — Query audio content (podcast)

With audio extraction enabled, the MP3 podcast is transcribed and indexed.
You can now ask questions about what was discussed in the recording.

In [ ]:
audio_queries = [
    'What topics are discussed in the podcast recording?',
    'What are the key points mentioned by the speakers?',
    'Summarize the main message from the audio content.',
]

print('=== Audio Content Queries (from MP3 transcription) ===')
print('These results come from audio extraction transcribing the podcast.\n')

for query in audio_queries:
    print(f'Q: {query}')
    response = kb.retrieve(query, num_results=3)
    for i, res in enumerate(response.get('retrievalResults', []), 1):
        score = res.get('score', 0)
        text = res['content']['text'][:180]
        uri = res.get('location', {}).get('s3Location', {}).get('uri', '')
        source = uri.split('/')[-1] if uri else ''
        print(f'  {i}. score={score:.4f} | source={source}')
        print(f'     {text}...')
    print()

## Step 7 — Query video content

With video extraction enabled, the M4V video is processed — audio is transcribed and visual content is described.

In [ ]:
video_queries = [
    'What is demonstrated or shown in the video?',
    'What are the key concepts explained in the video demo?',
    'Describe what happens in the video content.',
]

print('=== Video Content Queries (from M4V processing) ===')
print('These results come from video extraction processing the demo.\n')

for query in video_queries:
    print(f'Q: {query}')
    response = kb.retrieve(query, num_results=3)
    for i, res in enumerate(response.get('retrievalResults', []), 1):
        score = res.get('score', 0)
        text = res['content']['text'][:180]
        uri = res.get('location', {}).get('s3Location', {}).get('uri', '')
        source = uri.split('/')[-1] if uri else ''
        print(f'  {i}. score={score:.4f} | source={source}')
        print(f'     {text}...')
    print()

## Step 8 — Cross-modal AgenticRetrieveStream

Now the real power of multi-modal RAG: ask a question that requires synthesizing information across all three modalities.

AgenticRetrieveStream with `generate_response=True` will:
1. Decompose the query into sub-queries
2. Retrieve relevant chunks from PDF (text + image descriptions), audio transcript, and video content
3. Synthesize a unified answer with citations

In [ ]:
cross_modal_query = 'Based on all available content — the document, podcast, and video — what are the main topics covered and how do they relate to each other?'

print('=== Cross-Modal AgenticRetrieveStream ===')
print(f'Query: {cross_modal_query}\n')

result = kb.agentic_retrieve_stream(
    query=cross_modal_query,
    model_arn=generation_model_arn,
    generate_response=True,
    max_results=10,
    max_iterations=3,
)

if result.get('generated_response'):
    print(f"Generated Answer:\n{result['generated_response']['answer']}")
    citations = result['generated_response'].get('citations', [])
    print(f"\nChunks used: {len(result['results'])}")
    print(f"Citations: {len(citations)}")
else:
    print(f"Chunks retrieved: {len(result['results'])}")
    for i, r in enumerate(result['results'][:5], 1):
        print(f"  {i}. {r['content']['text'][:150]}...")

## Step 9 — Visual-specific generated answer

Ask specifically about chart/graph content. This demonstrates the value of image extraction — without it, the KB would have no knowledge about what's shown visually.

In [ ]:
visual_query = 'What do the charts, graphs, and figures in the document show? Describe the data, trends, and key numbers.'

print(f'Query: {visual_query}\n')

result = kb.agentic_retrieve_stream(
    query=visual_query,
    model_arn=generation_model_arn,
    generate_response=True,
)

if result.get('generated_response'):
    print(result['generated_response']['answer'])
    print(f"\nCitations: {len(result['generated_response'].get('citations', []))}")
else:
    print(f"Chunks: {len(result['results'])}")
    for i, r in enumerate(result['results'][:5], 1):
        print(f"  {i}. {r['content']['text'][:150]}...")

## Step 10 — Analyze chunk sources

Retrieve a broad set of chunks and group by source file to see what Advanced Indexing extracted from each modality.

In [ ]:
response = kb.retrieve('information data content technology', num_results=20)
chunks = response.get('retrievalResults', [])

print(f'=== Chunk Analysis by Source (total: {len(chunks)}) ===\n')

# Group by source file
from collections import defaultdict
source_chunks = defaultdict(list)

for chunk in chunks:
    uri = chunk.get('location', {}).get('s3Location', {}).get('uri', '')
    source_file = uri.split('/')[-1] if uri else 'unknown'
    source_chunks[source_file].append(chunk)

modality_icons = {'pdf': '📄 PDF', 'mp3': '🎵 AUDIO', 'm4v': '🎬 VIDEO', 'mp4': '🎬 VIDEO'}

for source, chunks_list in source_chunks.items():
    ext = source.split('.')[-1].lower() if '.' in source else ''
    icon = modality_icons.get(ext, '📎 OTHER')
    print(f'{icon}: {source} ({len(chunks_list)} chunks)')
    for i, chunk in enumerate(chunks_list[:2], 1):
        text = chunk['content']['text'][:150]
        score = chunk.get('score', 0)
        print(f'  {i}. score={score:.4f} | {text}...')
    if len(chunks_list) > 2:
        print(f'  ... and {len(chunks_list) - 2} more chunks')
    print()

## Step 11 — Cleanup

> Only run when done experimenting.

In [ ]:
# Uncomment to delete everything
# kb.delete_kb(delete_iam=True, delete_s3_bucket=True)

## Summary

### What we demonstrated

| Aspect | Details |
|---|---|
| **KB Type** | Managed — Bedrock handles the vector store |
| **Parsing** | Smart Parsing (always on) |
| **Advanced Indexing** | Image + Audio + Video extraction enabled |
| **Embedding** | Managed default (no extra cost) |
| **Data** | PDF (charts/tables), MP3 (podcast), M4V (video demo) |
| **Retrieval** | Retrieve (raw chunks), AgenticRetrieveStream (agentic + generation) |

### Smart Parsing vs Advanced Indexing

| Feature | Smart Parsing (always on) | Advanced Indexing (opt-in) |
|---------|--------------------------|---------------------------|
| Text from PDF | ✓ | — |
| Tables from PDF | ✓ (structure) | — |
| Charts/diagrams from PDF | ✗ (only text near them) | ✓ (describes visual content) |
| Audio transcription | Basic | ✓ (full transcription) |
| Video processing | Basic | ✓ (content extraction) |

### Configuration used

```python
kb = ManagedKnowledgeBase(
    kb_name='my-multimodal-kb',
    data_sources=[{
        'type': 'S3',
        'bucket_name': 'my-bucket',
        'enable_image_extraction': True,   # Charts/diagrams
        'enable_audio_extraction': True,   # MP3/WAV transcription
        'enable_video_extraction': True,   # MP4/M4V processing
    }],
    embedding_model=None,  # Managed default (no extra cost)
)
```

### Key takeaway

**Smart Parsing + Advanced Indexing** gives you true multi-modal RAG:
- No BDA (Bedrock Data Automation) pipelines needed
- No Amazon Transcribe setup for audio
- No separate image analysis service for charts
- No multimodal embedding model required
- Just enable the extraction toggles and query naturally

### Documentation

- [Advanced indexing for managed KBs](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-managed-advanced-indexing.html)
- [Supported file types](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-managed-file-types.html)
- [Create a managed knowledge base](https://docs.aws.amazon.com/bedrock/latest/userguide/kb-managed-create.html)